# NB03 — Evaluating Statistical Claims in Papers

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 2.5 hours  

---

## Motivation

Computational biology papers routinely report p-values, effect sizes, and comparisons across dozens of conditions. A researcher who cannot evaluate these claims critically is at the mercy of any paper they read — and at risk of citing work that is statistically unsound. This notebook builds four core capabilities: recognising NHST misinterpretation, computing statistical power, understanding effect size, and identifying p-value inflation from multiple testing.

## 1. NHST Misinterpretation: The Most Common Error

**Null Hypothesis Significance Testing (NHST)** asks: given that the null hypothesis is true, how probable is the observed data (or more extreme)? This probability is p.

### What p does NOT mean:

| Wrong interpretation | Why it's wrong |
|---------------------|----------------|
| "p = 0.03 means the null is probably false" | p is P(data | H₀), not P(H₀ | data). The latter requires a prior. |
| "p = 0.03 means the effect is real" | p only addresses evidence against H₀, not whether the effect is biologically meaningful |
| "p = 0.07 means no effect" | Failure to reject ≠ evidence for H₀. It may mean underpowered study. |
| "p < 0.05 is the threshold for truth" | 0.05 is an arbitrary convention, not a law of nature |

### What you should ask instead:
1. What is the **effect size**? (Cohen's d, odds ratio, etc.)
2. What is the **confidence interval** around the effect?
3. What was the **statistical power** of the study?
4. Were **multiple tests** performed? What correction was applied?

## 2. Statistical Power

**Power** = P(reject H₀ | H₁ is true) = the probability of detecting a real effect.

Power depends on:
- Effect size δ (larger effects are easier to detect)
- Sample size n (more samples → more power)
- Variance σ² (lower noise → higher power)
- Significance threshold α (lower α → lower power)

The required sample size for two-sample comparison:

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot 2\sigma^2}{\delta^2}$$

where $z_{\alpha/2}$ is the z-score for significance level α, $z_\beta$ is the z-score for power level (1-β), δ is the minimum detectable effect.

A **typical problem:** a paper reports p = 0.04 with n = 5 per group. If the study was underpowered (power < 50%), a true effect of this size has less than a 50% chance of being detected — and the effects that *are* detected are systematically **overestimated** (the "winner's curse").

## 3. Effect Size

| Measure | Formula | When to use | Interpretation |
|---------|---------|------------|----------------|
| Cohen's d | (μ₁ - μ₂) / σ_pooled | Two continuous groups | d=0.2 small, d=0.5 medium, d=0.8 large |
| Odds ratio | (a/b) / (c/d) | Binary outcome | OR > 1: exposure increases odds |
| Relative risk | P(outcome|exposed) / P(outcome|unexposed) | Cohort studies | RR = 2: doubles the risk |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)


def power_analysis(effect_size_d: float, alpha: float = 0.05,
                   power_target: float = 0.80) -> int:
    """Compute required sample size per group for a two-sample t-test.

    Uses the formula: n = (z_alpha/2 + z_beta)^2 * 2 / d^2
    where d is Cohen's effect size (delta / sigma).
    """
    z_alpha = stats.norm.ppf(1 - alpha / 2)  # two-tailed
    z_beta = stats.norm.ppf(power_target)
    n = ((z_alpha + z_beta) ** 2 * 2) / (effect_size_d ** 2)
    return int(np.ceil(n))


def cohens_d(group1: np.ndarray, group2: np.ndarray) -> float:
    """Compute Cohen's d for two independent groups."""
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1 - 1) * group1.std(ddof=1)**2 +
                          (n2 - 1) * group2.std(ddof=1)**2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std


def simulate_study(true_effect: float, n_per_group: int,
                   n_simulations: int = 1000, sigma: float = 1.0) -> dict:
    """Simulate many studies and track detection rate and mean detected effect."""
    detected_effects = []
    p_values = []
    for _ in range(n_simulations):
        g1 = np.random.normal(0, sigma, n_per_group)
        g2 = np.random.normal(true_effect, sigma, n_per_group)
        t_stat, p_val = stats.ttest_ind(g1, g2)
        p_values.append(p_val)
        if p_val < 0.05:
            detected_effects.append(abs(g2.mean() - g1.mean()))

    power_estimate = np.mean(np.array(p_values) < 0.05)
    mean_detected = np.mean(detected_effects) if detected_effects else 0
    return {
        'power': power_estimate,
        'n_detected': len(detected_effects),
        'true_effect': true_effect,
        'mean_detected_effect': mean_detected,
        'bias_ratio': mean_detected / true_effect if true_effect > 0 else 0,
    }


# Power analysis across effect sizes
effect_sizes = [0.2, 0.5, 0.8, 1.0, 1.5]
print("Required n per group for 80% power (two-sided t-test, alpha=0.05):")
print("-" * 50)
for d in effect_sizes:
    n_req = power_analysis(d)
    print(f"  Cohen's d = {d:.1f} ({['small','medium','large','large','very large'][effect_sizes.index(d)]}): "
          f"n = {n_req} per group")

# Winner's curse demonstration
print("\nWinner's curse (effect inflation in underpowered studies):")
for n in [5, 20, 100]:
    result = simulate_study(true_effect=0.5, n_per_group=n)
    print(f"  n={n:3d}: power={result['power']:.2f}, "
          f"true effect=0.50, mean detected={result['mean_detected_effect']:.2f}, "
          f"inflation={result['bias_ratio']:.2f}x")

In [ ]:
# P-hacking simulation: run 100 tests, report only significant ones
def simulate_phacking(n_tests: int = 100, alpha: float = 0.05,
                      n_per_group: int = 20, n_simulations: int = 10000) -> dict:
    """Simulate p-hacking: perform n_tests under null, report only significant.

    True false-positive rate under this practice equals 1 - (1-alpha)^n_tests.
    """
    false_positives = 0
    for _ in range(n_simulations):
        p_values = []
        for _ in range(n_tests):
            g1 = np.random.normal(0, 1, n_per_group)
            g2 = np.random.normal(0, 1, n_per_group)  # null: no true effect
            _, p = stats.ttest_ind(g1, g2)
            p_values.append(p)
        if min(p_values) < alpha:  # report the best result
            false_positives += 1

    observed_fpr = false_positives / n_simulations
    theoretical_fpr = 1 - (1 - alpha) ** n_tests
    return {'observed_fpr': observed_fpr, 'theoretical_fpr': theoretical_fpr}


# FDR vs Bonferroni comparison
def multiple_testing_comparison(n_tests: int = 50, n_true_effects: int = 10,
                                 effect_size: float = 0.8, n_per_group: int = 20):
    """Compare uncorrected, Bonferroni, and Benjamini-Hochberg (FDR) multiple testing."""
    from statsmodels.stats.multitest import multipletests

    p_values = []
    true_labels = []

    for i in range(n_tests):
        is_real = i < n_true_effects
        true_labels.append(is_real)
        g1 = np.random.normal(0, 1, n_per_group)
        effect = effect_size if is_real else 0
        g2 = np.random.normal(effect, 1, n_per_group)
        _, p = stats.ttest_ind(g1, g2)
        p_values.append(p)

    p_arr = np.array(p_values)
    true_arr = np.array(true_labels)

    _, bonf_rejected, _, _ = multipletests(p_arr, alpha=0.05, method='bonferroni')
    _, fdr_rejected, _, _ = multipletests(p_arr, alpha=0.05, method='fdr_bh')
    uncorr_rejected = p_arr < 0.05

    def stats_from_rejected(rejected):
        tp = np.sum(rejected & true_arr)
        fp = np.sum(rejected & ~true_arr)
        fn = np.sum(~rejected & true_arr)
        return {'TP': int(tp), 'FP': int(fp), 'FN': int(fn),
                'total_rejected': int(rejected.sum())}

    return {
        'uncorrected': stats_from_rejected(uncorr_rejected),
        'bonferroni': stats_from_rejected(bonf_rejected),
        'fdr_bh': stats_from_rejected(fdr_rejected),
    }


np.random.seed(42)
phack_result = simulate_phacking(n_tests=20)
print(f"P-hacking (20 tests, report best): observed FPR = {phack_result['observed_fpr']:.3f}, "
      f"theoretical = {phack_result['theoretical_fpr']:.3f}")
print(f"  (Expected single-test FPR without hacking: 0.050)")

mt_result = multiple_testing_comparison()
print("\nMultiple testing comparison (50 tests, 10 real effects, d=0.8):")
for method, s in mt_result.items():
    print(f"  {method:15s}: TP={s['TP']}, FP={s['FP']}, FN={s['FN']}, total_rejected={s['total_rejected']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Statistical Claim Evaluation — Power, Effect Size, Multiple Testing', fontsize=13, fontweight='bold')

# Panel 1: Power curve — n vs power at different effect sizes
ax1 = axes[0]
ns = np.arange(5, 200, 5)
z_alpha = stats.norm.ppf(0.975)  # two-tailed, alpha=0.05

for d, color, label in [(0.2, '#E53935', 'Small (d=0.2)'),
                          (0.5, '#FB8C00', 'Medium (d=0.5)'),
                          (0.8, '#43A047', 'Large (d=0.8)')]:
    se = np.sqrt(2 / ns)
    z_power = d / se - z_alpha
    power = stats.norm.cdf(z_power)
    ax1.plot(ns, power, color=color, linewidth=2, label=label)

ax1.axhline(0.80, color='gray', linestyle='--', alpha=0.7, label='80% power target')
ax1.set_xlabel('Sample size per group (n)')
ax1.set_ylabel('Statistical power')
ax1.set_title('Power Curve')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

# Panel 2: P-value distribution under null vs alternative
ax2 = axes[1]
np.random.seed(99)
n_sims = 5000
# Under null
p_null = [stats.ttest_ind(np.random.normal(0, 1, 30), np.random.normal(0, 1, 30))[1]
          for _ in range(n_sims)]
# Under alternative (d=0.5)
p_alt = [stats.ttest_ind(np.random.normal(0, 1, 30), np.random.normal(0.5, 1, 30))[1]
         for _ in range(n_sims)]

bins = np.linspace(0, 1, 21)
ax2.hist(p_null, bins=bins, alpha=0.6, color='steelblue', label='Under H₀ (no effect)', density=True)
ax2.hist(p_alt, bins=bins, alpha=0.6, color='salmon', label='Under H₁ (d=0.5, n=30)', density=True)
ax2.axvline(0.05, color='red', linestyle='--', linewidth=1.5, label='p=0.05 threshold')
ax2.set_xlabel('p-value')
ax2.set_ylabel('Density')
ax2.set_title('P-value Distribution:\nNull vs Alternative')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Panel 3: FDR vs uncorrected comparison
ax3 = axes[2]
methods_labels = ['Uncorrected', 'Bonferroni', 'FDR (BH)']
tp_vals = [mt_result['uncorrected']['TP'], mt_result['bonferroni']['TP'], mt_result['fdr_bh']['TP']]
fp_vals = [mt_result['uncorrected']['FP'], mt_result['bonferroni']['FP'], mt_result['fdr_bh']['FP']]

x = np.arange(len(methods_labels))
width = 0.35
ax3.bar(x - width/2, tp_vals, width, label='True Positives', color='#43A047', alpha=0.85)
ax3.bar(x + width/2, fp_vals, width, label='False Positives', color='#E53935', alpha=0.85)
ax3.set_xticks(x)
ax3.set_xticklabels(methods_labels)
ax3.set_ylabel('Count')
ax3.set_title('Multiple Testing Correction:\nTrue vs False Positives')
ax3.legend()
ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('statistical_claims.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

**Exercise 1:** A paper reports p = 0.04 with n = 8 per group and claims "the effect is statistically significant." Using `power_analysis()`, what effect size (Cohen's d) would this study have 50% power to detect? What does this imply about the reported result?

**Exercise 2:** A scRNA-seq paper performs differential expression analysis on 20,000 genes and reports 500 significant at p < 0.05 without multiple testing correction. How many false positives do you expect? What correction should they have applied?

**Exercise 3:** Compute Cohen's d for two groups: Group 1 = [2.1, 2.4, 1.9, 2.3, 2.0], Group 2 = [3.1, 2.8, 3.2, 2.9, 3.0]. Is this a small, medium, or large effect?

**Exercise 4 (reflection):** In one sentence: why can a p-value be significant but the effect be biologically unimportant?

## Quiz

1. What does p = 0.03 actually mean?
2. What is statistical power and what four factors affect it?
3. What is the winner's curse?
4. Why does Bonferroni correction reduce power more than Benjamini-Hochberg?
5. If a paper reports FDR < 0.05 for 200 genes, how many false positives are expected?

## References

- Cohen, J. (1992). A power primer. Psychological Bulletin 112(1):155–159.
- Ioannidis, J.P.A. (2005). Why most published research findings are false. PLOS Medicine 2(8):e124.
- Nuzzo, R. (2014). Statistical errors. Nature 506:150–152.